# 👑 KING2 + Qwen3.5-9B Fine-tuning
---
**تدريب Qwen3.5-9B على شخصية KING2 ومعرفة الرياضيات**

يقوم هذا النوت بوك بـ:
1. تحميل معرفة KING2 من `knowledge_base.json`
2. تحويلها إلى بيانات تدريب
3. تدريب Qwen3.5-9B باستخدام LoRA
4. حفظ الـ LoRA adapter في `qwen3.5-9b/lora_adapter`

⚠️ يحتاج GPU (T4 في Colab مجاني يكفي، لكن A100 أفضل لـ 9B)
⚠️ Qwen3.5-9B بحجم 9B معلمات → يحتاج 15GB VRAM مع 4-bit quantization

In [ ]:
# @title 1. تثبيت الاعتماديات
# Install required packages
!pip install -qU transformers datasets accelerate peft bitsandbytes
!pip install -qU trl
!pip install -qU --upgrade huggingface_hub

print('Dependencies installed!')

In [ ]:
# @title 2. تثبيت Unsloth (أسرع تدريب)
import torch
major_version, minor_version = torch.__version__.split('.')[:2]
print(f'PyTorch version: {torch.__version__}')

try:
    import unsloth
    print('Unsloth already installed!')
except:
    print('Installing Unsloth...')
    !pip install -qU unsloth
    !pip install -qU --force-reinstall --no-deps unsloth
    print('Unsloth installed!')

In [ ]:
# @title 3. تحميل بيانات KING2
import json
import os

from google.colab import files
print('⚠️ الرجاء رفع knowledge_base.json من مشروع KING2')
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'Loaded: {filename}')
    with open(filename, 'r', encoding='utf-8') as f:
        kb = json.load(f)
    break

# Extract math entries
math_cats = {'حساب', 'جبر', 'هندسة', 'إحصاء', 'تفاضل وتكامل', 'رياضيات'}
math_entries = [e for e in kb.get('knowledge', []) if e.get('category') in math_cats]
general_entries = [e for e in kb.get('knowledge', []) if e.get('category') not in math_cats]

print(f'Math entries: {len(math_entries)}')
print(f'General entries: {len(general_entries)}')
print(f'Total: {len(kb["knowledge"])}')

In [ ]:
# @title 4. إعداد بيانات التدريب (KING2 Format for Qwen3.5)
import pandas as pd

# شخصية KING2 - هوية النظام
system_prompt = (
    'أنت KING2، مساعد الذكاء الاصطناعي الملكي المتطور المبني على Qwen3.5-9B. '
    'أجب بالعربية الفصحى أولاً دائماً. '
    'كن مختصراً ومفيداً وواضحاً. '
    'استخدم markdown عند الحاجة. '
    'للمسائل الرياضية: اشرح الخطوات بالتفصيل. '
    'إذا سألك المستخدم بلغة أخرى، أجب بنفس اللغة.'
)

def create_chat_format(question, answer):
    """Convert to Qwen3.5 chat format with system prompt."""
    return {
        'system': system_prompt,
        'input': question,
        'output': answer
    }

# إنشاء بيانات التدريب
training_data = []

# 1. المدخلات الرياضية
for entry in math_entries:
    training_data.append(create_chat_format(
        entry['question'],
        entry['answer']
    ))

# 2. مدخلات عامة (شخصية KING2)
general_qa = [
    {'q': 'من أنت؟', 'a': 'أنا KING2، مساعد الذكاء الاصطناعي الملكي المتطور المبني على Qwen3.5-9B. أنا هنا لمساعدتك في أي شيء تحتاجه.'},
    {'q': 'ما اسمك؟', 'a': 'اسمي KING2، ويعني الملك الذهبي. أنا مساعد ذكي تم تطويره لخدمتك، مدعوم بتقنية Qwen3.5.'},
    {'q': 'بأي لغة تتحدث؟', 'a': 'أتحدث العربية أولاً وبشكل أساسي، ولكن يمكنني التحدث بالإنجليزية والصينية والعديد من اللغات الأخرى.'},
    {'q': 'ماذا تستطيع أن تفعل؟', 'a': 'أستطيع مساعدتك في: الحساب والرياضيات، البرمجة، التحليل، الترجمة، الكتابة الإبداعية، البحث في الإنترنت، تحليل الصور، واستخدام الأدوات.'},
    {'q': 'من صنعك؟', 'a': 'أنا KING2، تم تطويري من قبل فريق KING2 AI، مبني على Qwen3.5-9B ليكون أول مساعد ذكاء اصطناعي عربي متكامل.'},
    {'q': 'ما هي مجالات خبرتك؟', 'a': 'خبرتي تشمل الرياضيات (الجبر، الهندسة، التفاضل والتكامل، الإحصاء)، البرمجة (Python، JavaScript)، الترجمة، التحليل، البحث، والرؤية الحاسوبية.'},
    {'q': 'كيف تقوم بحل المسائل الرياضية؟', 'a': 'أقوم بحل المسائل الرياضية خطوة بخطوة، أشرح كل خطوة بالتفصيل مع الأمثلة لضمان الفهم الكامل.'},
    {'q': 'هل يمكنك مساعدتي في الواجبات المدرسية؟', 'a': 'نعم، يمكنني مساعدتك في فهم المفاهيم وحل التمارين. اشرح لي السؤال وسأبسطه لك خطوة بخطوة.'},
    {'q': 'ما هي سعة الذاكرة لديك؟', 'a': 'أمتلك سعة سياقية تصل إلى 262,144 رمزاً، مما يسمح لي بمعالجة محادثات طويلة جداً ومستندات كبيرة.'},
    {'q': 'هل تدعم الرؤية الحاسوبية؟', 'a': 'نعم، أدعم تحليل الصور والرؤية الحاسوبية محلياً، يمكنني وصف الصور وتحليل محتواها مباشرة.'},
]

for item in general_qa:
    training_data.append(create_chat_format(item['q'], item['a']))

print(f'Training samples: {len(training_data)}')

In [ ]:
# @title 5. تجهيز البيانات للـ Trainer (Qwen3.5 Chat Format)
from datasets import Dataset

def format_qwen_chat(example):
    """Convert to Qwen3.5 chat template format using tokenizer's chat_template."""
    messages = [
        {'role': 'system', 'content': example['system']},
        {'role': 'user', 'content': example['input']},
        {'role': 'assistant', 'content': example['output']}
    ]
    # We'll format with the tokenizer after loading it
    return {'messages': messages}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_qwen_chat)

print(f'Dataset size: {len(dataset)}')
print(f'Example messages: {dataset[0]["messages"]}')

In [ ]:
# @title 6. تحميل Qwen3.5-9B
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Qwen3.5-9B from HuggingFace
model_name = 'Qwen/Qwen3.5-9B'  # or lmstudio-community variant

# 4-bit quantization لتوفير الذاكرة
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print(f'Loading {model_name} in 4-bit...')
print(f'This may take a few minutes for a 9B model...')

try:
    # Try with Unsloth first (faster)
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        load_in_4bit=True,
        max_seq_length=4096,
    )
    print('Loaded with Unsloth!')
    unsloth_available = True
except Exception as e:
    print(f'Unsloth failed: {e}')
    print('Loading with standard Transformers...')
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )
    tokenizer.padding_side = 'right'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map='auto',
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )
    unsloth_available = False

print(f'Model loaded! Parameters: {model.num_parameters() / 1e9:.2f}B')

In [ ]:
# @title 7. تنسيق البيانات مع tokenizer
def apply_chat_template(example):
    """Apply Qwen3.5's chat template to format the conversation."""
    formatted = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': formatted}

dataset = dataset.map(apply_chat_template)

print(f'Formatted example:')
print(dataset[0]['text'][:500])

In [ ]:
# @title 8. إعداد LoRA
from peft import LoraConfig, get_peft_model, TaskType

# Qwen3.5 target modules (similar to Qwen2.5 architecture)
target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                  'gate_proj', 'up_proj', 'down_proj']

if unsloth_available:
    from unsloth import is_bfloat16_supported
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.1,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
        use_rslora=True,
        max_seq_length=4096
    )
else:
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.1,
        bias='none',
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)

print('LoRA configured!')
print(f'Trainable parameters: {model.num_parameters(only_trainable=True) / 1e6:.2f}M')

In [ ]:
# @title 9. التدريب
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./king2-qwen3.5-9b-lora',
    num_train_epochs=3,
    per_device_train_batch_size=1,  # 9B needs smaller batch
    gradient_accumulation_steps=8,
    warmup_steps=10,
    logging_steps=5,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    save_strategy='epoch',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
    max_seq_length=4096,
    dataset_text_field='text',
    packing=False,
)

print('Starting training...')
print(f'Batch size: {training_args.per_device_train_batch_size}')
print(f'Gradient accumulation: {training_args.gradient_accumulation_steps}')
print(f'Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'Epochs: {training_args.num_train_epochs}')
print('\n⚠️  هذا قد يستغرق 30-60 دقيقة على T4')

trainer.train()

print('\n✅ Training complete!')

In [ ]:
# @title 10. حفظ النموذج
import shutil
import os

# حفظ LoRA adapter
model.save_pretrained('./king2-qwen3.5-9b-lora/final')
tokenizer.save_pretrained('./king2-qwen3.5-9b-lora/final')

# إنشاء مجلد مضغوط للتحميل
!zip -r king2-qwen-lora.zip ./king2-qwen3.5-9b-lora/

print('\n✅ Model saved!')
print('📁 LoRA adapter: ./king2-qwen3.5-9b-lora/final')
print('📦 Zip: king2-qwen-lora.zip')
print('\n⬇️ تحميل الملف:')
from google.colab import files
files.download('king2-qwen-lora.zip')

In [ ]:
# @title 11. تجربة النموذج
def ask_king2(prompt):
    if unsloth_available:
        FastLanguageModel.for_inference(model)
    
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': prompt}
    ]
    
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted, return_tensors='pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract assistant response
    if 'assistant' in response:
        response = response.split('assistant')[-1]
    return response.strip()

# اختبارات
tests = [
    'من أنت؟',
    'ما هو الجمع في الرياضيات؟',
    'اشرح لي نظرية فيثاغورس',
    'اكتب دالة بلغة Python لجمع رقمين',
]

for test in tests:
    print(f'\n❓ {test}')
    result = ask_king2(test)
    print(f'👑 {result[:300]}...' if len(result) > 300 else f'👑 {result}')
    print('---')

In [ ]:
# @title 12. (اختياري) دمج LoRA مع النموذج الأساسي
# هذا الخيار يدمج LoRA مع النموذج الأصلي لتوليد GGUF يمكن استخدامه مع LM Studio

merge = input('هل تريد دمج LoRA مع النموذج الأصلي؟ (y/n): ')

if merge.lower() == 'y':
    print('Merging LoRA with base model...')
    print('⚠️  هذا قد يستغرق 10-15 دقيقة لـ 9B model')
    
    # Merge
    merged_model = model.merge_and_unload()
    
    # Save merged
    merged_model.save_pretrained('./king2-qwen3.5-9b-merged', safe_serialization=True)
    tokenizer.save_pretrained('./king2-qwen3.5-9b-merged')
    
    # Zip for download
    !zip -r king2-qwen-merged.zip ./king2-qwen3.5-9b-merged/
    
    print('\n✅ Merged model saved!')
    print('📁 Path: ./king2-qwen3.5-9b-merged/')
    
    # Download
    from google.colab import files
    files.download('king2-qwen-merged.zip')
else:
    print('Skipping merge. You can use the LoRA adapter separately.')

---
## 📝 ملخص

**الخطوات التالية بعد التدريب:**
1. حمل الملف `king2-qwen-lora.zip` من Colab إلى جهازك
2. فك الضغط وانسخ مجلد `lora_adapter` إلى `qwen3.5-9b/lora_adapter/` في مشروع KING2
3. KING2 engine سيكتشف الـ adapter تلقائياً عند تشغيل Qwen3.5-9B
4. اختبر النموذج مع أسئلة الرياضيات والشخصية

**الملفات المنتجة:**
- LoRA adapter: `./king2-qwen3.5-9b-lora/`
- Merged model: `./king2-qwen3.5-9b-merged/` (اختياري)

**لمساعدة إضافية:** راسلني في المحادثة!